# RAG Smoke Test Notebook

This notebook assumes you have already built persistent **Chroma** indexes (e.g., via `build_indexes.py`) under:
- `data/indexes/TPMS`
- `data/indexes/AA`

It will:
1) Load your local retriever (`rag.py`)  
2) Run **retrieval-only** tests (inspect top-k chunks)  
3) Run **full RAG** tests (retrieve → answer with citations)

In [11]:
import os
import time
from dotenv import load_dotenv

# Load environment variables (expects OPENAI_API_KEY)
try:
    load_dotenv()
except Exception as e:
    print("dotenv not available or failed to load:", e)
assert os.getenv(
    "OPENAI_API_KEY"), "OPENAI_API_KEY is missing. Set it in your environment or in a .env file."
print("OPENAI_API_KEY is set ✅")

OPENAI_API_KEY is set ✅


In [12]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
print("Running from:", HERE)

def is_repo_root(p: Path) -> bool:
    # Prefer strong layout markers to avoid accidentally selecting /scripts as root
    return (
        (p / "pyproject.toml").exists()
        or (p / ".git").exists()
        or ((p / "app").exists() and (p / "scripts").exists() and (p / "data").exists())
        or ((p / "knowledge").exists() and (p / "data").exists())
        or (p / "main.py").exists()
    )

PROJECT_ROOT = next(p for p in [HERE, *HERE.parents] if is_repo_root(p))

INDEX_BASE = PROJECT_ROOT / "data" / "indexes"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("INDEX_BASE:", INDEX_BASE)
print("INDEX_BASE exists?", INDEX_BASE.exists())

# Make repo root importable (optional, but harmless)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


Running from: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/scripts
PROJECT_ROOT: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot
INDEX_BASE: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/data/indexes
INDEX_BASE exists? True


In [13]:
# Import rag.py reliably (even if app/ isn't a python package)
import sys
from pathlib import Path

RAG_DIR = PROJECT_ROOT / "app" / "nodes"
if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))

import rag  # app/nodes/rag.py

print("Imported rag.py from:", rag.__file__)
print("Project root:", PROJECT_ROOT)
print("Embedding model:", getattr(rag, "EMBED_MODEL", None))


Imported rag.py from: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/app/nodes/rag.py
Project root: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot
Embedding model: text-embedding-3-small


In [18]:
PRODUCTS = ["TPMS", "AA"]
DEFAULT_K = 5

missing = [p for p in PRODUCTS if not (INDEX_BASE / p).exists()]
if missing:
    print("⚠️ Missing index folders for:", missing)
    print("Expected paths like:", [str(INDEX_BASE / p) for p in missing])
else:
    print("All index folders found ✅")


# Simple queries to use for smoke tests (edit freely)
SMOKE_QUERIES = {
    "TPMS": [
        "Cómo se rotan los sensores y cuándo conviene hacerlo?",
    ],
    "AA": [
        "Cuánto consumo tiene un aire 2200 24V aproximadamente?",
    ],
}

All index folders found ✅


In [15]:
# Helper: robust retrieval call across LangChain versions
def pretty_doc(doc, max_chars: int = 420) -> str:
    meta = getattr(doc, "metadata", {}) or {}
    src = meta.get("source") or meta.get("file_path") or meta.get("path") or ""
    page = meta.get("page", meta.get("page_number", ""))
    head = f"source={src} page={page}".strip()
    txt = (getattr(doc, "page_content", "") or "").strip().replace("\n", " ")
    if len(txt) > max_chars:
        txt = txt[:max_chars] + "…"
    return f"{head}\n{txt}"

## 1) Retrieval-only smoke test

This prints the **top-k chunks** returned by the retriever so you can validate:
- The index loads correctly
- The docs look relevant
- Metadata looks reasonable (source paths, etc.)


In [ ]:
for product_id in PRODUCTS:
    print("\n" + "="*90)
    print("RETRIEVAL TEST:", product_id)
    print("="*90)
    print(product_id)
    retriever = rag.get_retriever(product_id=product_id, k=DEFAULT_K)
    print(retriever)
    queries = SMOKE_QUERIES.get(product_id) #, ["Explain the main troubleshooting steps."])

    for q in queries:
        t0 = time.perf_counter()
        docs = retriever.invoke(q)
        dt = (time.perf_counter() - t0) * 1000

        print(f"\nQ: {q}")
        print(f"Retrieved {len(docs)} docs in {dt:.1f} ms")
        for i, d in enumerate(docs, 1):
            print(f"\n--- doc {i} ---")
            print(pretty_doc(d))


## 2) Full RAG test (retrieve → answer)

This uses:
- `rag.get_retriever()` for retrieval
- `ChatOpenAI` for generation
- A simple prompt that forces citation markers like `[1] [2]` pointing to retrieved chunks


In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

def format_context(docs, max_chars_each: int = 1200) -> str:
    blocks = []
    for i, d in enumerate(docs, 1):
        meta = getattr(d, "metadata", {}) or {}
        src = meta.get("source") or meta.get("file_path") or meta.get("path") or f"chunk_{i}"
        txt = (getattr(d, "page_content", "") or "").strip()
        if len(txt) > max_chars_each:
            txt = txt[:max_chars_each] + "…"
        blocks.append(f"[{i}] SOURCE: {src}\n{txt}")
    return "\n\n".join(blocks)

SYSTEM_PROMPT = '''You are a technical support assistant.
Answer the user's question using ONLY the provided context.
If the context is insufficient, say so and ask 1-2 precise follow-up questions.
Always include citations like [1], [2] pointing to the context chunks you used.
Answer in Spanish (Rioplatense, clear and practical).'''

def rag_answer(product_id: str, question: str, k: int = 5, model: str = "gpt-4o-mini", temperature: float = 0.0):
    retriever = rag.get_retriever(product_id=product_id, k=k)

    t0 = time.perf_counter()
    docs = retriever.invoke(question)
    t_retr = (time.perf_counter() - t0) * 1000

    context = format_context(docs)

    llm = ChatOpenAI(model=model, temperature=temperature)

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"PRODUCT: {product_id}\n\nCONTEXT:\n{context}\n\nQUESTION:\n{question}")
    ]

    t1 = time.perf_counter()
    resp = llm.invoke(messages)
    t_gen = (time.perf_counter() - t1) * 1000

    return {
        "answer": getattr(resp, "content", str(resp)),
        "docs": docs,
        "timing_ms": {"retrieve": t_retr, "generate": t_gen, "total": t_retr + t_gen}
    }


In [19]:
# Run a couple of RAG examples per product
for product_id in PRODUCTS:
    print("\n" + "="*90)
    print("RAG TEST:", product_id)
    print("="*90)

    for q in SMOKE_QUERIES.get(product_id, [])[:2]:
        out = rag_answer(product_id, q, k=DEFAULT_K)
        print(f"\nQ: {q}")
        print("Timing (ms):", out["timing_ms"])
        print("\nA:\n", out["answer"])

        # Show the sources returned
        print("\nSources returned (top-k):")
        for i, d in enumerate(out["docs"], 1):
            meta = getattr(d, "metadata", {}) or {}
            print(f"  [{i}] {meta.get('source') or meta.get('file_path') or meta.get('path') or 'unknown'}")



RAG TEST: TPMS

Q: Cómo se rotan los sensores y cuándo conviene hacerlo?
Timing (ms): {'retrieve': 458.26270800898783, 'generate': 3568.0134579888545, 'total': 4026.2761659978423}

A:
 Para rotar los sensores de un sistema TPMS, es importante seguir la guía específica para cada modelo. La rotación de neumáticos se recomienda hacer cada 5,000 a 8,000 kilómetros, o según lo indique el fabricante del vehículo. Esto ayuda a asegurar un desgaste uniforme de los neumáticos y a mantener un rendimiento óptimo del sistema TPMS.

Puedes consultar las guías de rotación de neumáticos para cada modelo en los siguientes enlaces:

- [Rotación de neumáticos C240](https://tpms.com.ar/soporte-c260/#rotacion)
- [Rotación de neumáticos C260](https://tpms.com.ar/soporte-c260/#rotacion)
- [Rotación de neumáticos C101](https://tpms.com.ar/soporte-c101/#rotacion)
- [Rotación de neumáticos C270](https://tpms.com.ar/soporte-c270/#rotacion) [4].

Sources returned (top-k):
  [1] /Users/damiangarayalde/Work - Git